<a href="https://colab.research.google.com/github/nitajadav8/Educational-Video-Summarization-in-Indian-Languages_5minSegment-based-approach/blob/main/EvaluatingTranscripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas matplotlib seaborn tqdm

In [2]:
!pip install https://github.com/kpu/kenlm/archive/master.zip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.6/553.6 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kenlm: filename=kenlm-0.2.0-cp312-cp312-linux_x86_64.whl size=3187996 sha256=2f223098f0f38465b9ecd79d7374c778f9d2d9a37974ef139b08cf5a0baa00ed
  Stored in directory: /tmp/pip-ephem-wheel-cache-e5e_5jxw/wheels/92/c8/12/56d187154e078f0eaa74d059017fc1afe1c4d91fbce02ce8d9
Successfully built kenlm


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import re
import kenlm

# ==============================
# CONFIG
# ==============================
CSV_PATH = "transcripts.csv"
TEXT_COLUMN = "transcript"
MODEL_PATH = "model.arpa"
OUTPUT_CSV = "perplexity_output.csv"


def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.dropna(subset=[TEXT_COLUMN])
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)
    return df

# ==============================
# TEXT CLEANING
# ==============================
def clean_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

# ==============================
# LOAD LANGUAGE MODEL
# ==============================
def load_lm(model_path):
    model = kenlm.Model(model_path)
    return model

# ==============================
# PERPLEXITY CALCULATION
# ==============================
def compute_perplexity(model, text):
    """
    KenLM returns log score → convert to perplexity
    """
    text = clean_text(text)

    if len(text.split()) < 2:
        return np.nan

    log_score = model.score(text, bos=True, eos=True)
    word_count = len(text.split())

    perplexity = np.exp(-log_score / word_count)
    return perplexity

# ==============================
# PROCESS DATA
# ==============================
def process(df, model):
    perplexities = []

    for text in tqdm(df[TEXT_COLUMN]):
        ppl = compute_perplexity(model, text)
        perplexities.append(ppl)

    df["perplexity"] = perplexities
    return df

# ==============================
# ANALYSIS
# ==============================
def analyze(df):
    print("\n=== BASIC STATS ===")
    print(df["perplexity"].describe())

    print("\n=== HIGH PERPLEXITY SAMPLES ===")
    print(df.sort_values("perplexity", ascending=False)[[TEXT_COLUMN, "perplexity"]].head(10))

# ==============================
# VISUALIZATION
# ==============================
def visualize(df):
    sns.set(style="whitegrid")

    # Histogram
    plt.figure()
    sns.histplot(df["perplexity"].dropna(), bins=50)
    plt.title("Perplexity Distribution")
    plt.xlabel("Perplexity")
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot
    plt.figure()
    sns.boxplot(x=df["perplexity"].dropna())
    plt.title("Perplexity Boxplot")
    plt.show()

    # Perplexity over index
    plt.figure()
    plt.plot(df["perplexity"].values)
    plt.title("Perplexity over Segments")
    plt.xlabel("Segment Index")
    plt.ylabel("Perplexity")
    plt.show()


def main():
    print("Loading data...")
    df = load_csv(CSV_PATH)

    print("Loading language model...")
    model = load_lm(MODEL_PATH)

    print("Computing perplexity...")
    df = process(df, model)

    print("Saving results...")
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    analyze(df)
    visualize(df)

if __name__ == "__main__":
    main()